# ROGII - Wellbore Geology Prediction

![ROGII Wellbore Geology Prediction](https://www.googleapis.com/download/storage/v1/b/kaggle-forum-message-attachments/o/inbox%2F4080021%2F3f56527c733365a94d929bdc0600c7ef%2Fig_023b4ba06ac0441e0169fa9248ca54819aacb93888a02601a8.png?generation=1778029361497538&alt=media)

This Kaggle notebook is structured as an EDA-first workflow for the ROGII Wellbore Geology Prediction competition. The core objective is to predict `TVT` (True Vertical Thickness) for the hidden interval of each horizontal well.

The public sample run indicates a file-per-well problem rather than a single-table tabular competition: 773 train wells, 3 public test wells, one horizontal CSV and one typewell CSV per well, and `sample_submission.csv` rows keyed by `{WELLNAME}_{row_index}`. The public test set is intentionally tiny; hidden reruns can change the test inventory, so the notebook computes most insights dynamically.

Workflow:

1. Resolve the Kaggle data path and establish consistent plotting style.
2. Discover files and validate the submission index.
3. Build lightweight well-level metadata without loading the entire dataset into one large table.
4. Inspect schema, missingness, and hidden evaluation windows.
5. Plot representative horizontal wells against their typewell reference logs.
6. Review target/feature relationships on a manageable training sample.
7. Create a simple carry-forward baseline submission for a valid starting point.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)
sns.set_theme(style='whitegrid', context='notebook', palette='viridis')
VIRIDIS = sns.color_palette('viridis', as_cmap=True)
VIRIDIS_COLORS = sns.color_palette('viridis', 8)

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
COMPETITION_SLUG = 'rogii-wellbore-geology-prediction'
DATA_ROOT_CANDIDATES = [
    KAGGLE_INPUT_ROOT / 'competitions' / COMPETITION_SLUG,
    KAGGLE_INPUT_ROOT / COMPETITION_SLUG,
]


def resolve_data_root(candidates):
    for candidate in candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate
    for sample_file in KAGGLE_INPUT_ROOT.rglob('sample_submission.csv') if KAGGLE_INPUT_ROOT.exists() else []:
        if COMPETITION_SLUG in sample_file.as_posix():
            return sample_file.parent
    return candidates[0]


DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
WORK_DIR = Path('/kaggle/working')
SUBMISSION_PATH = WORK_DIR / 'submission.csv'

print('Kaggle input root:', KAGGLE_INPUT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('Exists:', DATA_ROOT.exists())
print('sample_submission exists:', (DATA_ROOT / 'sample_submission.csv').exists())


def fmt_int(value):
    if pd.isna(value):
        return 'n/a'
    return f'{int(value):,}'


def fmt_float(value, digits=2):
    if pd.isna(value):
        return 'n/a'
    return f'{float(value):,.{digits}f}'


def fmt_pct(value, digits=1):
    if pd.isna(value):
        return 'n/a'
    return f'{100 * float(value):,.{digits}f}%'


def show_insights(title, bullets):
    clean_bullets = [str(bullet) for bullet in bullets if bullet]
    body = '\n'.join(f'- {bullet}' for bullet in clean_bullets)
    display(Markdown(f'### {title}\n{body}'))


## 1. Data Path And File Discovery

This section confirms that Kaggle mounted the competition data correctly, then inventories the available files. The competition uses a nested file layout:

- horizontal wells contain measured depth (`MD`), coordinates, gamma ray (`GR`), and `TVT_input`;
- typewells provide the vertical `GR` signature indexed by `TVT`, plus geology labels;
- `sample_submission.csv` identifies the exact horizontal-well rows that require prediction.

In the saved public-sample run, the dataset contains 773 training horizontal wells, 773 training typewells, 3 public test horizontal wells, 3 public test typewells, and 773 PNG reference images. The hidden test rerun can expose more wells, so the notebook avoids hardcoded well IDs.

In [ ]:
def find_files(root: Path, pattern: str):
    return sorted(root.rglob(pattern)) if root.exists() else []


def well_name_from_horizontal_path(path: Path) -> str:
    return path.name.split('__horizontal_well.csv')[0]


def well_name_from_typewell_path(path: Path) -> str:
    return path.name.split('__typewell.csv')[0]


def parse_submission_id(value):
    well, row = str(value).rsplit('_', 1)
    return well, int(row)


def get_column(df: pd.DataFrame, name: str):
    lookup = {col.lower(): col for col in df.columns}
    return lookup.get(name.lower())


train_dir = DATA_ROOT / 'train'
test_dir = DATA_ROOT / 'test'
sample_path = DATA_ROOT / 'sample_submission.csv'

train_horizontal_files = find_files(train_dir, '*__horizontal_well.csv')
train_typewell_files = find_files(train_dir, '*__typewell.csv')
test_horizontal_files = find_files(test_dir, '*__horizontal_well.csv')
test_typewell_files = find_files(test_dir, '*__typewell.csv')
png_files = find_files(DATA_ROOT, '*.png')

inventory = pd.DataFrame([
    {'split': 'train', 'file_type': 'horizontal_well', 'count': len(train_horizontal_files)},
    {'split': 'train', 'file_type': 'typewell', 'count': len(train_typewell_files)},
    {'split': 'test', 'file_type': 'horizontal_well', 'count': len(test_horizontal_files)},
    {'split': 'test', 'file_type': 'typewell', 'count': len(test_typewell_files)},
    {'split': 'all', 'file_type': 'png', 'count': len(png_files)},
    {'split': 'all', 'file_type': 'sample_submission', 'count': int(sample_path.exists())},
])
display(inventory)
print('First train wells:', [well_name_from_horizontal_path(p) for p in train_horizontal_files[:5]])
print('First test wells:', [well_name_from_horizontal_path(p) for p in test_horizontal_files[:5]])

In [ ]:
if 'inventory' in globals() and not inventory.empty:
    counts = inventory.set_index(['split', 'file_type'])['count']
    train_wells = counts.get(('train', 'horizontal_well'), 0)
    test_wells = counts.get(('test', 'horizontal_well'), 0)
    png_count = counts.get(('all', 'png'), 0)
    show_insights('Discovery Insights', [
        f'Train inventory contains {fmt_int(train_wells)} horizontal wells and matching typewell files, giving a broad set of well-level examples for validation.',
        f'Public test inventory contains {fmt_int(test_wells)} horizontal wells. Kaggle code competitions can rerun on a larger hidden test set, so all downstream logic is file-discovery based.',
        f'The {fmt_int(png_count)} PNG reference images are useful for qualitative QA, but the notebook keeps modeling inputs to CSV features that are available at inference time.',
        f'Data root resolved to `{DATA_ROOT}`; `sample_submission.csv` exists: {(DATA_ROOT / "sample_submission.csv").exists()}.'
    ])

### 1.1 Parse The Submission Index

`sample_submission.csv` is more than an output template; it defines the exact evaluation rows. Splitting each `id` into `well` and `row_idx` gives an immediate check that the requested predictions align with the hidden suffix of each horizontal well.

In [ ]:
sample_submission = pd.read_csv(sample_path)
id_col = sample_submission.columns[0]
target_col = 'tvt' if 'tvt' in sample_submission.columns else sample_submission.columns[-1]

submission_index = sample_submission[id_col].map(parse_submission_id)
sample_submission['well'] = [x[0] for x in submission_index]
sample_submission['row_idx'] = [x[1] for x in submission_index]

display(sample_submission.head())
print('sample_submission shape:', sample_submission.shape)
display(sample_submission.groupby('well')['row_idx'].agg(['min', 'max', 'count']).head(10))

In [ ]:
if 'sample_submission' in globals() and not sample_submission.empty:
    sub_summary = sample_submission.groupby('well')['row_idx'].agg(['min', 'max', 'count'])
    show_insights('Submission Index Insights', [
        f'The public sample requests {fmt_int(len(sample_submission))} predictions across {fmt_int(sample_submission["well"].nunique())} wells.',
        f'Per-well requested rows range from {fmt_int(sub_summary["count"].min())} to {fmt_int(sub_summary["count"].max())}, so the hidden interval length varies by well.',
        f'The first requested row is usually the first hidden `TVT_input` row; mismatches here would signal an indexing or path issue before submission generation.',
    ])

## 2. Load Lightweight Metadata

This section reads each horizontal well and typewell once to build compact, well-level summaries. The goal is not to model yet; it is to understand the shape of the problem with enough detail to avoid mistakes later.

Key fields captured here:

- row counts and measured-depth ranges, which describe lateral length;
- `GR` mean and standard deviation, which show how log character varies across wells;
- `TVT_input` known and missing counts, which reveal the size of the hidden interval;
- train-only `TVT` range, which gives a rough sense of target movement by well;
- typewell `TVT` range and geology label counts, which describe the reference logs used for correlation.

In the saved public-sample run, the public test wells are actually drawn from train-like examples, making them useful for notebook validation but not necessarily representative of the hidden leaderboard set.

In [ ]:
def summarize_horizontal_file(path: Path, split: str):
    well = well_name_from_horizontal_path(path)
    df = pd.read_csv(path)
    tvt_input_col = get_column(df, 'TVT_input')
    tvt_col = get_column(df, 'TVT')
    md_col = get_column(df, 'MD')
    gr_col = get_column(df, 'GR')

    row = {
        'split': split,
        'well': well,
        'rows': len(df),
        'n_columns': df.shape[1],
        'columns': tuple(df.columns),
        'has_tvt': tvt_col is not None,
        'has_tvt_input': tvt_input_col is not None,
        'md_min': pd.to_numeric(df[md_col], errors='coerce').min() if md_col else np.nan,
        'md_max': pd.to_numeric(df[md_col], errors='coerce').max() if md_col else np.nan,
        'gr_mean': pd.to_numeric(df[gr_col], errors='coerce').mean() if gr_col else np.nan,
        'gr_std': pd.to_numeric(df[gr_col], errors='coerce').std() if gr_col else np.nan,
    }

    if tvt_input_col:
        tvt_input = pd.to_numeric(df[tvt_input_col], errors='coerce')
        hidden_mask = tvt_input.isna()
        row['tvt_input_known'] = int(tvt_input.notna().sum())
        row['tvt_input_missing'] = int(hidden_mask.sum())
        row['tvt_input_missing_frac'] = float(hidden_mask.mean())
        row['first_missing_row'] = int(np.argmax(hidden_mask.to_numpy())) if hidden_mask.any() else np.nan
        row['last_known_tvt_input'] = float(tvt_input.ffill().iloc[-1]) if tvt_input.notna().any() else np.nan
    else:
        row['tvt_input_known'] = 0
        row['tvt_input_missing'] = np.nan
        row['tvt_input_missing_frac'] = np.nan
        row['first_missing_row'] = np.nan
        row['last_known_tvt_input'] = np.nan

    if tvt_col:
        tvt = pd.to_numeric(df[tvt_col], errors='coerce')
        row['tvt_min'] = tvt.min()
        row['tvt_max'] = tvt.max()
        row['tvt_range'] = tvt.max() - tvt.min()
    else:
        row['tvt_min'] = np.nan
        row['tvt_max'] = np.nan
        row['tvt_range'] = np.nan

    return row


def summarize_typewell_file(path: Path, split: str):
    well = well_name_from_typewell_path(path)
    df = pd.read_csv(path)
    tvt_col = get_column(df, 'TVT')
    gr_col = get_column(df, 'GR')
    geology_col = get_column(df, 'Geology')
    return {
        'split': split,
        'well': well,
        'rows': len(df),
        'n_columns': df.shape[1],
        'columns': tuple(df.columns),
        'tvt_min': pd.to_numeric(df[tvt_col], errors='coerce').min() if tvt_col else np.nan,
        'tvt_max': pd.to_numeric(df[tvt_col], errors='coerce').max() if tvt_col else np.nan,
        'gr_mean': pd.to_numeric(df[gr_col], errors='coerce').mean() if gr_col else np.nan,
        'gr_std': pd.to_numeric(df[gr_col], errors='coerce').std() if gr_col else np.nan,
        'n_geology_labels': df[geology_col].nunique() if geology_col else np.nan,
    }


horizontal_meta = pd.DataFrame(
    [summarize_horizontal_file(p, 'train') for p in train_horizontal_files]
    + [summarize_horizontal_file(p, 'test') for p in test_horizontal_files]
)
typewell_meta = pd.DataFrame(
    [summarize_typewell_file(p, 'train') for p in train_typewell_files]
    + [summarize_typewell_file(p, 'test') for p in test_typewell_files]
)

display(horizontal_meta.head())
display(typewell_meta.head())

In [ ]:
if 'horizontal_meta' in globals() and not horizontal_meta.empty:
    train_meta = horizontal_meta.query("split == 'train'")
    test_meta = horizontal_meta.query("split == 'test'")
    bullets = []
    if not train_meta.empty:
        bullets.append(f'Train wells have a median of {fmt_int(train_meta["rows"].median())} horizontal rows and a median hidden `TVT_input` fraction of {fmt_pct(train_meta["tvt_input_missing_frac"].median())}.')
        bullets.append(f'Train `TVT` range varies substantially by well: median range {fmt_float(train_meta["tvt_range"].median())} ft, max range {fmt_float(train_meta["tvt_range"].max())} ft.')
    if not test_meta.empty:
        bullets.append(f'Public test wells have a median of {fmt_int(test_meta["rows"].median())} rows and median hidden fraction of {fmt_pct(test_meta["tvt_input_missing_frac"].median())}.')
    if 'typewell_meta' in globals() and not typewell_meta.empty:
        bullets.append(f'Typewells provide compact vertical references with median {fmt_int(typewell_meta["rows"].median())} rows and median {fmt_int(typewell_meta["n_geology_labels"].median())} geology labels.')
    show_insights('Well Metadata Insights', bullets)

## 3. Schema, Missingness, And Evaluation Window

This section checks which columns are available in train versus test and summarizes missingness. This is the most important leakage-control step in the notebook.

Observed from the saved public-sample run:

- train horizontal wells include `TVT` plus geology-top columns such as `ANCC`, `ASTNU`, `ASTNL`, `EGFDU`, `EGFDL`, and `BUDA`;
- test horizontal wells expose only inference-time columns such as `MD`, `X`, `Y`, `Z`, `GR`, and `TVT_input`;
- approximately 73% of each horizontal well is hidden in `TVT_input`, so the task is closer to sequence continuation / log correlation than ordinary row-wise regression.

Modeling implication: train-only geology-top columns should not be used directly as features for a final test-time model unless they are predicted by a separate model that can also run on test data.

In [ ]:
def column_presence(files, split, well_parser):
    rows = []
    for path in files:
        well = well_parser(path)
        df = pd.read_csv(path, nrows=5)
        for col in df.columns:
            rows.append({'split': split, 'well': well, 'column': col})
    return pd.DataFrame(rows)


horizontal_columns = pd.concat([
    column_presence(train_horizontal_files, 'train', well_name_from_horizontal_path),
    column_presence(test_horizontal_files, 'test', well_name_from_horizontal_path),
], ignore_index=True)

column_summary = (
    horizontal_columns.groupby(['split', 'column'])['well']
    .nunique()
    .reset_index(name='well_count')
    .sort_values(['split', 'well_count', 'column'], ascending=[True, False, True])
)
display(column_summary)

if not horizontal_meta.empty:
    display(horizontal_meta.groupby('split')[['rows', 'tvt_input_missing_frac', 'tvt_range', 'gr_mean', 'gr_std']].describe().T)

### 3.1 Well-Level Distribution Plots

These plots compare well length, hidden-window size, and typewell coverage across splits. They are quick checks for public-test representativeness and for validation design: if test wells sit in the same range as train wells, masked-tail validation is more trustworthy.

In [ ]:
if not horizontal_meta.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    sns.histplot(data=horizontal_meta, x='rows', hue='split', bins=30, ax=axes[0], element='step', palette='viridis')
    axes[0].set_title('Rows per horizontal well')
    sns.histplot(data=horizontal_meta, x='tvt_input_missing_frac', hue='split', bins=30, ax=axes[1], element='step', palette='viridis')
    axes[1].set_title('Hidden TVT_input fraction')
    sns.scatterplot(data=horizontal_meta, x='rows', y='tvt_input_missing', hue='split', ax=axes[2], palette='viridis')
    axes[2].set_title('Hidden rows by well length')
    plt.tight_layout()
    plt.show()

if not typewell_meta.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.histplot(data=typewell_meta, x='rows', hue='split', bins=30, ax=axes[0], element='step', palette='viridis')
    axes[0].set_title('Rows per typewell')
    sns.histplot(data=typewell_meta, x='n_geology_labels', hue='split', bins=20, ax=axes[1], element='step', palette='viridis')
    axes[1].set_title('Geology labels per typewell')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'horizontal_meta' in globals() and not horizontal_meta.empty:
    by_split = horizontal_meta.groupby('split')['tvt_input_missing_frac'].median()
    show_insights('Missingness And Window Insights', [
        f'Median hidden `TVT_input` fraction by split: ' + ', '.join([f'{idx}: {fmt_pct(val)}' for idx, val in by_split.items()]) + '.',
        'The hidden interval is long enough that pure carry-forward is a weak geological model; it is mainly a submission smoke test.',
        'Because test lacks `TVT` and formation-top columns, final models should be trained only on inference-available features or on auxiliary predictions that are also generated for test.',
    ])

## 4. Representative Well Inspection

These plots inspect a few evaluation wells at the well-log level. Each row of plots is designed to answer a modeling question:

- Does the horizontal `GR` curve contain recognizable peaks and troughs that can be aligned to the typewell?
- Where does `TVT_input` stop, and how long is the hidden interval?
- Does the typewell cover the same `TVT` depth range needed by the horizontal well?

The visual pattern to look for is local similarity between the horizontal `GR` trace and the typewell `GR` trace after mapping the horizontal row/MD position to TVT. This is the domain reason that alignment-style models can be more promising than plain tabular models.

In [ ]:
horizontal_lookup = {
    **{well_name_from_horizontal_path(p): p for p in train_horizontal_files},
    **{well_name_from_horizontal_path(p): p for p in test_horizontal_files},
}
typewell_lookup = {
    **{well_name_from_typewell_path(p): p for p in train_typewell_files},
    **{well_name_from_typewell_path(p): p for p in test_typewell_files},
}


def load_well(well):
    horizontal = pd.read_csv(horizontal_lookup[well])
    typewell = pd.read_csv(typewell_lookup[well]) if well in typewell_lookup else None
    return horizontal, typewell


def plot_well(well):
    horizontal, typewell = load_well(well)
    md_col = get_column(horizontal, 'MD')
    gr_col = get_column(horizontal, 'GR')
    tvt_input_col = get_column(horizontal, 'TVT_input')
    tvt_col = get_column(horizontal, 'TVT')

    x = pd.to_numeric(horizontal[md_col], errors='coerce') if md_col else horizontal.index
    fig, axes = plt.subplots(1, 3, figsize=(19, 4))

    if gr_col:
        axes[0].plot(x, pd.to_numeric(horizontal[gr_col], errors='coerce'), lw=1, color=VIRIDIS_COLORS[2])
    axes[0].set_title(f'{well}: horizontal GR')
    axes[0].set_xlabel('MD' if md_col else 'row')
    axes[0].set_ylabel('GR')

    if tvt_input_col:
        tvt_input = pd.to_numeric(horizontal[tvt_input_col], errors='coerce')
        axes[1].plot(x, tvt_input, lw=1, label='TVT_input', color=VIRIDIS_COLORS[4])
        if tvt_input.isna().any():
            hidden_start = int(np.argmax(tvt_input.isna().to_numpy()))
            axes[1].axvline(x.iloc[hidden_start] if hasattr(x, 'iloc') else hidden_start, color=VIRIDIS_COLORS[7], ls='--', lw=1, label='first hidden row')
    if tvt_col:
        axes[1].plot(x, pd.to_numeric(horizontal[tvt_col], errors='coerce'), lw=1, alpha=0.65, label='TVT', color=VIRIDIS_COLORS[1])
    axes[1].invert_yaxis()
    axes[1].set_title('Horizontal TVT track')
    axes[1].set_xlabel('MD' if md_col else 'row')
    axes[1].legend(loc='best')

    if typewell is not None:
        tw_tvt_col = get_column(typewell, 'TVT')
        tw_gr_col = get_column(typewell, 'GR')
        geology_col = get_column(typewell, 'Geology')
        if tw_tvt_col and tw_gr_col:
            axes[2].plot(pd.to_numeric(typewell[tw_gr_col], errors='coerce'), pd.to_numeric(typewell[tw_tvt_col], errors='coerce'), lw=1, color=VIRIDIS_COLORS[3])
            axes[2].invert_yaxis()
        if geology_col:
            label_counts = typewell[geology_col].value_counts().head(8)
            axes[2].text(0.98, 0.02, '\n'.join([f'{k}: {v}' for k, v in label_counts.items()]), transform=axes[2].transAxes, ha='right', va='bottom', fontsize=8)
    axes[2].set_title('Typewell GR vs TVT')
    axes[2].set_xlabel('GR')
    axes[2].set_ylabel('TVT')

    plt.tight_layout()
    plt.show()


sample_wells = list(sample_submission['well'].drop_duplicates().head(3))
if len(sample_wells) < 3:
    sample_wells += [w for w in horizontal_lookup if w not in sample_wells][:3 - len(sample_wells)]

for well in sample_wells:
    if well in horizontal_lookup:
        plot_well(well)

## 5. Train Target And Feature Relationships

This section samples training rows to study numeric relationships without forcing the whole dataset into memory at once. The focus is on `TVT`, `TVT_input`, `MD`, `Z`, `GR`, and the train-only formation-top columns.

Saved-run observations:

- the training sample contains 90,000 rows from 60 wells;
- `TVT_input` is populated only in the observed part of each well, so it has far fewer non-null sampled rows than `TVT`;
- `GR` has substantial missingness in the sampled rows, so models need an explicit missing-data strategy;
- formation-top columns are highly informative in train but absent from test, making them a leakage risk if used naively.

Useful next modeling directions include lag/rolling features over `GR`, slope features over `TVT_input`, typewell-correlation features, and per-well normalization of log values.

In [ ]:
def load_horizontal_sample(files, max_wells=60, rows_per_well=1500, split='train'):
    frames = []
    for path in files[:max_wells]:
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        if len(df) > rows_per_well:
            df = df.sample(rows_per_well, random_state=42).sort_index()
        df = df.copy()
        df['well'] = well
        df['split'] = split
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


train_sample = load_horizontal_sample(train_horizontal_files)
print('train sample shape:', train_sample.shape)
display(train_sample.head())

numeric_cols = [c for c in ['MD', 'X', 'Y', 'Z', 'GR', 'TVT', 'TVT_input', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA'] if c in train_sample.columns]
if numeric_cols:
    display(train_sample[numeric_cols].describe().T)
    corr = train_sample[numeric_cols].apply(pd.to_numeric, errors='coerce').corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap='viridis', annot=False, square=True)
    plt.title('Train numeric correlation sample')
    plt.tight_layout()
    plt.show()

### 5.1 Target Distribution And Scatter Checks

The following plots show the sampled target distribution and coarse relationships between `TVT`, `GR`, and `MD`. The goal is not to prove a linear model; it is to see whether simple global relationships exist or whether the problem needs per-well, sequence-aware features.

In [ ]:
if not train_sample.empty and 'TVT' in train_sample.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    sns.histplot(pd.to_numeric(train_sample['TVT'], errors='coerce'), bins=60, ax=axes[0], color=VIRIDIS_COLORS[4])
    axes[0].set_title('Train TVT distribution')
    if 'GR' in train_sample.columns:
        sns.scatterplot(data=train_sample.sample(min(len(train_sample), 20000), random_state=42), x='GR', y='TVT', hue='well', legend=False, s=8, alpha=0.35, ax=axes[1], palette='viridis')
        axes[1].invert_yaxis()
        axes[1].set_title('GR vs TVT sample')
    if 'MD' in train_sample.columns:
        sns.scatterplot(data=train_sample.sample(min(len(train_sample), 20000), random_state=42), x='MD', y='TVT', hue='well', legend=False, s=8, alpha=0.35, ax=axes[2], palette='viridis')
        axes[2].invert_yaxis()
        axes[2].set_title('MD vs TVT sample')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'train_sample' in globals() and not train_sample.empty and 'TVT' in train_sample.columns:
    numeric_train = train_sample.apply(pd.to_numeric, errors='coerce')
    gr_missing = numeric_train['GR'].isna().mean() if 'GR' in numeric_train else np.nan
    tvt_input_missing = numeric_train['TVT_input'].isna().mean() if 'TVT_input' in numeric_train else np.nan
    corr_md_tvt = numeric_train[['MD', 'TVT']].corr().iloc[0, 1] if {'MD', 'TVT'}.issubset(numeric_train.columns) else np.nan
    corr_gr_tvt = numeric_train[['GR', 'TVT']].corr().iloc[0, 1] if {'GR', 'TVT'}.issubset(numeric_train.columns) else np.nan
    show_insights('Target Relationship Insights', [
        f'The sampled train table has {fmt_int(len(train_sample))} rows across {fmt_int(train_sample["well"].nunique())} wells.',
        f'`GR` missingness in this sample is {fmt_pct(gr_missing)}, so missingness flags and interpolation choices may matter.',
        f'`TVT_input` missingness in this sample is {fmt_pct(tvt_input_missing)}, consistent with a known-prefix / hidden-suffix task.',
        f'Correlation in the sample: `MD` vs `TVT` = {fmt_float(corr_md_tvt, 3)}, `GR` vs `TVT` = {fmt_float(corr_gr_tvt, 3)}. Correlation alone is not enough; the useful signal is likely local log-shape alignment.',
    ])

## 6. EDA Summary And Modeling Hypotheses

Main takeaways from the public-sample EDA:

- The task is well-level sequence completion: each well has a known `TVT_input` prefix and a hidden suffix to predict.
- The public test sample mirrors three training wells, but the hidden test rerun can be larger and structurally different.
- The evaluation window is long, around three quarters of each public-sample well, so errors can compound if a model only extrapolates from the final known point.
- `GR` is the main test-time signal for geological correlation, especially when compared against the typewell `GR` signature indexed by `TVT`.
- Train-only geology tops are useful for analysis and auxiliary targets, but not safe as direct inference features.
- A constant carry-forward baseline is valid and surprisingly stable on smooth wells, but it cannot capture faults, bed-boundary jumps, or gradual TVT drift.

Modeling backlog after EDA:

1. Build a stronger validation split by masking the tail of train wells in the same way as `TVT_input`.
2. Add slope and curvature features from the known `TVT_input` prefix.
3. Add rolling-window `GR` features and missingness flags.
4. Explore typewell alignment features using cross-correlation or dynamic time warping-like search.
5. Compare simple per-well extrapolation, tree models, and sequence models under the same tail-mask validation protocol.

## 7. Baseline Submission

The baseline intentionally stays simple: fill the hidden interval by carrying forward the last known `TVT_input` value for each well, then map predictions back to the requested `sample_submission.csv` rows.

Why keep this baseline:

- it validates the full Kaggle submission path end to end;
- it gives a sanity-check score for future modeling work;
- it is hard to break because it depends only on test-time columns;
- it highlights exactly where better models need to add value: predicting TVT drift and discontinuities after the known prefix.

In the saved public-sample sanity check, masking the last 25% of the first 40 training wells produced an average well-level RMSE of about 12.54. Treat that as an internal smoke-test number, not a leaderboard estimate.

In [ ]:
def carry_forward_tvt(horizontal: pd.DataFrame) -> pd.Series:
    tvt_input_col = get_column(horizontal, 'TVT_input')
    tvt_col = get_column(horizontal, 'TVT')

    if tvt_input_col is not None:
        tvt = pd.to_numeric(horizontal[tvt_input_col], errors='coerce')
    elif tvt_col is not None:
        tvt = pd.to_numeric(horizontal[tvt_col], errors='coerce')
    else:
        tvt = pd.Series(np.nan, index=horizontal.index, dtype='float64')

    filled = tvt.ffill().bfill()
    if filled.isna().all():
        filled = pd.Series(0.0, index=horizontal.index)
    return filled.astype('float64')


def make_well_predictions(horizontal_files):
    predictions = {}
    for path in horizontal_files:
        well = well_name_from_horizontal_path(path)
        horizontal = pd.read_csv(path)
        predictions[well] = carry_forward_tvt(horizontal).reset_index(drop=True)
    return predictions


well_predictions = make_well_predictions(test_horizontal_files)
print('prediction wells:', len(well_predictions))

### 7.1 Masked-Tail Baseline Validation

This quick validation hides the tail of training wells and asks the carry-forward baseline to predict it. The setup approximates the competition structure and gives a first reference score for future feature engineering.

In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))) if mask.any() else np.nan


def carry_forward_validation(horizontal_files, tail_fraction=0.25, max_wells=40):
    scores = []
    for path in horizontal_files[:max_wells]:
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        tvt_col = get_column(df, 'TVT')
        tvt_input_col = get_column(df, 'TVT_input')
        if tvt_col is None:
            continue

        y = pd.to_numeric(df[tvt_col], errors='coerce')
        x = pd.to_numeric(df[tvt_input_col], errors='coerce') if tvt_input_col else y.copy()
        eval_start = int(len(df) * (1 - tail_fraction))
        x.iloc[eval_start:] = np.nan

        tmp = df.copy()
        tmp['TVT_input'] = x
        pred = carry_forward_tvt(tmp)
        scores.append({'well': well, 'rows': len(df), 'rmse': rmse(y.iloc[eval_start:], pred.iloc[eval_start:])})

    return pd.DataFrame(scores)


if train_horizontal_files:
    val_scores = carry_forward_validation(train_horizontal_files)
    display(val_scores.head())
    print('Mean well RMSE:', val_scores['rmse'].mean())
else:
    print('No train files found in this environment.')

In [ ]:
if 'val_scores' in globals() and not val_scores.empty:
    show_insights('Baseline Validation Insights', [
        f'Carry-forward validation mean RMSE is {fmt_float(val_scores["rmse"].mean())} across {fmt_int(len(val_scores))} train wells.',
        f'Best/worst validation wells range from {fmt_float(val_scores["rmse"].min())} to {fmt_float(val_scores["rmse"].max())} RMSE, showing that some wells are smooth while others need true geological alignment.',
        'Use this result as a smoke-test baseline. A stronger model should be compared under the same masked-tail validation setup before leaderboard submission.',
    ])

### 7.2 Write The Submission File

The final cell maps per-well predictions back into the original `sample_submission.csv` order. The quality checks confirm row count, prediction range, and whether any requested well was missing from the discovered test files.

In [ ]:
submission = sample_submission[[id_col]].copy()
predicted_tvt = []
missing_wells = set()

global_fallback = 0.0
all_known_values = [series.dropna().to_numpy() for series in well_predictions.values()]
if all_known_values:
    non_empty_values = [values for values in all_known_values if len(values)]
    if non_empty_values:
        global_fallback = float(np.nanmedian(np.concatenate(non_empty_values)))

for sub_id in sample_submission[id_col]:
    well, row_idx = parse_submission_id(sub_id)
    pred_series = well_predictions.get(well)
    if pred_series is None or len(pred_series) == 0:
        missing_wells.add(well)
        predicted_tvt.append(global_fallback)
    elif 0 <= row_idx < len(pred_series):
        predicted_tvt.append(float(pred_series.iloc[row_idx]))
    else:
        predicted_tvt.append(float(pred_series.iloc[-1]))

submission[target_col] = predicted_tvt
submission.to_csv(SUBMISSION_PATH, index=False)

print('Wrote:', SUBMISSION_PATH)
print('Rows:', len(submission))
print('Missing wells:', sorted(missing_wells)[:10], 'count=', len(missing_wells))
display(submission.head())
display(submission[target_col].describe())

In [ ]:
if 'submission' in globals() and not submission.empty:
    show_insights('Submission Output Insights', [
        f'Generated {fmt_int(len(submission))} predictions and wrote `{SUBMISSION_PATH}`.',
        f'Prediction range is {fmt_float(submission[target_col].min())} to {fmt_float(submission[target_col].max())} ft, with median {fmt_float(submission[target_col].median())} ft.',
        f'Missing-well fallback count is {fmt_int(len(missing_wells))}; this should be zero for a clean Kaggle run.',
    ])